# Chapter 4 — Semantic Search

Interactive companion to `chapters/chapter_04.qmd`. Run Chapter 1's batch extraction first if you haven't:

```bash
python code/chapter_01/read_ddr.py datasets/sample_ddrs/ --batch --out datasets/ddr_text
```

This notebook downloads the `all-MiniLM-L6-v2` embedding model (~90MB) on first run.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "code" / "chapter_04"))

# SentenceTransformer is a pretrained neural network that turns text into a
# vector of numbers (an "embedding") capturing its meaning. MODEL_NAME picks
# which pretrained model to load; the other functions come from our own
# code/chapter_04/semantic_search.py.
from sentence_transformers import SentenceTransformer
from semantic_search import MODEL_NAME, load_chunks, embed_texts, search

TEXT_DIR = Path.cwd().parent / "datasets" / "ddr_text"

# Downloads the model the first time this runs (~90MB), then reuses the
# local copy on every run after that.
model = SentenceTransformer(MODEL_NAME)

# load_chunks() reads every .txt report and returns two parallel lists:
# filenames[i] is the name of the report whose text is texts[i].
filenames, texts = load_chunks(TEXT_DIR)

# embed_texts() runs every report's text through the model, producing one
# embedding vector per report.
embeddings = embed_texts(model, texts)
print(f"Embedded {len(filenames)} reports")

In [ ]:
# search() embeds the query the same way, then ranks every report by how
# close its embedding is to the query's (cosine similarity) -- "close"
# meaning semantically similar, not just sharing the same words.
results = search(model, "stuck pipe", filenames, embeddings, top_k=10)

# enumerate(results, start=1) numbers the results starting at 1 instead of the
# usual 0, so the printed ranking reads "1st, 2nd, 3rd..." for people.
for rank, (name, score) in enumerate(results, start=1):
    print(f"{rank}. {score:.4f}  {name}")

## Try it yourself

Confirm report #39 (`FORGE-16A-78-32_Drilling_039_2020-11-27.txt`) shows up somewhere in this ranking, unlike Chapter 3's keyword search where it didn't appear at all.

In [ ]:
# List comprehension: build a new list containing just the filename from
# each (name, score) pair in results, discarding the score. The underscore in
# `_score` is a Python convention meaning "this variable is unused".
ranked_names = [name for name, _score in results]

assert "FORGE-16A-78-32_Drilling_039_2020-11-27.txt" in ranked_names

# .index() finds the position of an item in a list (counting from 0), so we
# add 1 to report it the way a person would ("rank 1" rather than "rank 0").
rank = ranked_names.index("FORGE-16A-78-32_Drilling_039_2020-11-27.txt") + 1
print(f"Confirmed: report #39 found at rank {rank} of {len(ranked_names)}")